# 02 — Feature Engineering

Builds customer-level features from raw transactions and saves to `data/processed/customers.csv`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RANDOM_SEED = 42
DATA_DIR = Path('..') / 'data' / 'raw'

pd.set_option('display.max_columns', 100)


In [2]:
df = pd.read_csv(DATA_DIR / 'ecommerce_customer_data_large.csv')
df.columns = df.columns.str.lower().str.replace(' ', '_')
df['purchase_date'] = pd.to_datetime(df['purchase_date'])
print(f'Raw data loaded: {df.shape}')


Raw data loaded: (250000, 13)


## 1. Build Customer-Level Features & Save

In [3]:
from scipy import stats

df_sorted = df.sort_values(['customer_id', 'purchase_date']).copy()
df_sorted['transaction_value'] = df_sorted['product_price'] * df_sorted['quantity']

DATASET_MAX_DATE = df['purchase_date'].max()

def avg_days_between(dates):
    if len(dates) < 2:
        return np.nan
    return dates.diff().dt.days.dropna().mean()

def std_days_between(dates):
    if len(dates) < 3:
        return np.nan
    return dates.diff().dt.days.dropna().std()

def spend_trend(values):
    if len(values) < 3:
        return np.nan
    slope, *_ = stats.linregress(range(len(values)), values)
    return slope

# Core customer-level aggregations
customers = df_sorted.groupby('customer_id').agg(
    earliest_transaction_date = ('purchase_date',     'min'),
    latest_transaction_date   = ('purchase_date',     'max'),
    n_transactions            = ('purchase_date',     'count'),
    unique_product_categories = ('product_category',  'nunique'),
    avg_amount_spent          = ('transaction_value', 'mean'),
    std_amount_spent          = ('transaction_value', 'std'),
    total_spend               = ('transaction_value', 'sum'),
    total_returns             = ('returns',           'sum'),
    n_payment_methods_used    = ('payment_method',    'nunique'),
    female                    = ('gender',            lambda x: int((x == 'Female').iloc[0])),
    customer_age              = ('customer_age',      'first'),
    churn                     = ('churn',             'first'),
)

# Recency, tenure, return rate
customers['days_since_last_purchase'] = (DATASET_MAX_DATE - customers['latest_transaction_date']).dt.days
customers['customer_tenure_days']     = (customers['latest_transaction_date'] - customers['earliest_transaction_date']).dt.days
customers['return_rate']              = customers['total_returns'] / customers['n_transactions']

# Purchase frequency (transactions per day of tenure; fallback for single-day customers)
customers['purchase_frequency'] = customers['n_transactions'] / customers['customer_tenure_days'].replace(0, np.nan)

# Payment method — % of transactions
pay_dummies = pd.get_dummies(df_sorted['payment_method'], prefix='pay')
pay_totals  = pay_dummies.join(df_sorted['customer_id']).groupby('customer_id').sum()
pay_pcts    = pay_totals.div(customers['n_transactions'], axis=0)
pay_pcts.columns = ['prct_paid_cash', 'prct_paid_credit_card', 'prct_paid_paypal']
customers = customers.join(pay_pcts)

# Product category — % of transactions
cat_dummies      = pd.get_dummies(df_sorted['product_category'], prefix='prct_txn')
cat_txn_totals   = cat_dummies.join(df_sorted['customer_id']).groupby('customer_id').sum()
cat_txn_pcts     = cat_txn_totals.div(customers['n_transactions'], axis=0)
customers        = customers.join(cat_txn_pcts)

# Product category — % of money spent
cat_spend = df_sorted.groupby(['customer_id', 'product_category'])['transaction_value'].sum().unstack(fill_value=0)
cat_spend.columns = [f'prct_spend_{c.lower()}' for c in cat_spend.columns]
cat_spend_pcts = cat_spend.div(customers['total_spend'], axis=0)
customers = customers.join(cat_spend_pcts)
customers = customers.drop(columns='total_spend')

# Time-between-transactions stats
avg_time = (
    df_sorted.groupby('customer_id')['purchase_date']
    .apply(avg_days_between)
    .rename('avg_days_between_transactions')
)
std_time = (
    df_sorted.groupby('customer_id')['purchase_date']
    .apply(std_days_between)
    .rename('std_days_between_transactions')
)
customers = customers.join(avg_time).join(std_time)

# Spend trend (slope of transaction values over time)
trend = (
    df_sorted.groupby('customer_id')['transaction_value']
    .apply(spend_trend)
    .rename('spend_trend')
)
customers = customers.join(trend)

print(customers.shape)
customers.head()
# Save processed customer-level dataset
PROCESSED_DIR = Path('..') / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
customers.to_csv(PROCESSED_DIR / 'customers.csv')
print(f'Saved {customers.shape[0]} customers to {PROCESSED_DIR / "customers.csv"}')


(49661, 29)
Saved 49661 customers to ../data/processed/customers.csv
